In [1]:
import os

os.environ["PYSPARK_PYTHON"] = ".venv/bin/python"
os.environ["PYSPARK_DRIVER_PYTHON"] = ".venv/bin/python"
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("iceberg-maintenance-experiments")
    .config("spark.jars.packages", ",".join([
        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.7.0",
        "org.apache.iceberg:iceberg-aws-bundle:1.7.0",
        "org.apache.hadoop:hadoop-aws:3.3.4",
    ]))
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.glue", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.glue.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
    .config("spark.sql.catalog.glue.warehouse", "s3://binance-iceberg-lake/warehouse")
    .config("spark.sql.catalog.glue.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.parquet.enableVectorizedReader", "false")
    .config("spark.sql.iceberg.vectorization.enabled", "false")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "3g")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

:: loading settings :: url = jar:file:/home/ubuntu/binance-iceberg-lakehouse/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ubuntu/.ivy2/cache
The jars for the packages stored in: /home/ubuntu/.ivy2/jars
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-096ee310-7283-47ef-b1e0-03a9acf88303;1.0
	confs: [default]
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.7.0 in central
	found org.apache.iceberg#iceberg-aws-bundle;1.7.0 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 279ms :: artifacts dl 8ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.apache.iceberg#iceberg-aws-bundle;1.7.0 from cent

In [2]:
from pathlib import Path
import pandas as pd
import time

RESULT_DIR = Path("results")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

def sql_df(query: str):
    return spark.sql(query)

def show_pd(query: str):
    return spark.sql(query).toPandas()

def timed_sql(query: str):
    start = time.time()
    df = spark.sql(query)
    rows = df.collect()
    elapsed = time.time() - start
    return rows, elapsed

def save_markdown(df: pd.DataFrame, path: str, title: str):
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with out.open("w", encoding="utf-8") as f:
        f.write(f"# {title}\n\n")
        f.write(df.to_markdown(index=False))
        f.write("\n")

In [4]:
target = "glue.binance_lakehouse.mor_orders_lab"
before_manifest = show_pd(f"""
SELECT COUNT(*) AS manifest_count
FROM {target}.manifests
""")

before_manifest

,manifest_count
0,5


In [5]:
spark.sql("""
CALL glue.system.rewrite_manifests(
  table => 'binance_lakehouse.mor_orders_lab'
)
""").show(truncate=False)

26/05/07 17:52:57 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------------------+---------------------+
|rewritten_manifests_count|added_manifests_count|
+-------------------------+---------------------+
|5                        |2                    |
+-------------------------+---------------------+



In [6]:
after_manifest = show_pd(f"""
SELECT COUNT(*) AS manifest_count
FROM {target}.manifests
""")

after_manifest

,manifest_count
0,2


In [7]:
result = pd.DataFrame([
    {"stage": "before", "manifest_count": int(before_manifest.iloc[0]["manifest_count"])},
    {"stage": "after", "manifest_count": int(after_manifest.iloc[0]["manifest_count"])},
])

save_markdown(
    result,
    "results/rewrite_manifests_result.md",
    "Rewrite Manifests Result"
)

result

,stage,manifest_count
0,before,5
1,after,2
